In [ ]:
import os
import cv2
import git
import numpy as np
import random
from glob import glob
from pathlib import Path


# =======================
# GLOBAL PROJECT ROOT
# =======================
ROOT_DIR = Path(git.Repo(".", search_parent_directories=True).working_tree_dir)

# =======================
# CONFIG (relative to repo)
# =======================
INPUT_DIR = ROOT_DIR / "raw-data" / "pastis" / "uncleaned"
REMOVE_TXT = ROOT_DIR / "dataset-preparation" / "pastis" / "removed_data.txt"
OUTPUT_DIR = ROOT_DIR / "raw-data" / "full-pastis-RGB-jitter"

PREFIX = "test"
SUFFIX = ".png"
START_INDEX = 0

# =======================
# LOAD REMOVE LIST
# =======================
with open(REMOVE_TXT, "r", encoding="utf-8") as f:
    remove_set = {line.strip() for line in f if line.strip()}

# =======================
# LIST FILES (all files in folder)
# =======================
files = sorted(
    f for f in os.listdir(INPUT_DIR)
    if os.path.isfile(INPUT_DIR / f)
)

# =======================
# RENAME (2-STEP TO AVOID COLLISIONS)
# =======================
temp_files = []
for i, old_name in enumerate(files):
    tmp_name = f"__tmp__{i}__{old_name}"
    os.rename(INPUT_DIR / old_name, INPUT_DIR / tmp_name)
    temp_files.append(tmp_name)

final_names = []
for i, tmp_name in enumerate(temp_files):
    new_name = f"{PREFIX}{START_INDEX + i}{SUFFIX}"
    os.rename(INPUT_DIR / tmp_name, INPUT_DIR / new_name)
    final_names.append(new_name)

# =======================
# DELETE FILES IN TXT (names assumed like test14.png)
# =======================
deleted = 0
for fname in remove_set:
    p = INPUT_DIR / fname
    if p.exists():
        p.unlink()
        deleted += 1

print(f"Renamed: {len(final_names)} | Deleted: {deleted}")


Scanning cleaned directory for .npz files: D:\Download\PASTIS\PASTIS\full-pastis-RGB
Found 1590 unique base names from .npz files in the cleaned directory.
Output directory for filtered PNG dataset: D:\Download\PASTIS\PASTIS\Final Cleaned

Scanning source directory: D:\Download\PASTIS\PASTIS\Complete Dataset
Copying .png files whose base names match those found in the cleaned directory...
------------------------------
------------------------------

File copying process finished.
Total source files matching pattern 'test*.png' found: 2433
Files copied to 'D:\Download\PASTIS\PASTIS\Final Cleaned' (matching base names in cleaned dir): 1590

--- Script Complete ---
The original source directory remains unchanged.
Filtered PNG files are in: D:\Download\PASTIS\PASTIS\Final Cleaned
-----------------------


In [ ]:
# Index-based batches (based on digits in filename) -- Corresponds to the original batches in the dataset: S2_1XXXX, S2_2XXXX, S2_3XXXX, S2_4XXXX
BATCH_RANGES = {
    "batch0": (0, 530),
    "batch1": (531, 1153),
    "batch2": (1154, 1876),
    "batch3": (1877, 2432),
}

IMG_EXTS = (".jpg", ".jpeg", ".png")


# =======================
# HELPERS
# =======================
def extract_index(filename):
    """Extract integer index from a filename like test123.png."""
    digits = "".join(filter(str.isdigit, filename))
    return int(digits) if digits else None


def get_batches(image_folder):
    """Return dict: batch_name -> list of image paths."""
    imgs = []
    for ext in IMG_EXTS:
        imgs += glob(os.path.join(str(image_folder), f"*{ext}"))

    batches = {name: [] for name in BATCH_RANGES}

    for p in imgs:
        fname = os.path.basename(p)
        num = extract_index(fname)
        if num is None:
            continue

        for name, (lo, hi) in BATCH_RANGES.items():
            if lo <= num <= hi:
                batches[name].append(p)
                break

    return batches


def calculate_batch_stats(img_paths):
    """Compute (mean,std) for (R,G,B) over per-image channel means."""
    r_means, g_means, b_means = [], [], []

    for p in img_paths:
        img = cv2.imread(p)
        if img is None:
            continue

        b, g, r = cv2.split(img.astype(np.float32))
        r_means.append(r.mean())
        g_means.append(g.mean())
        b_means.append(b.mean())

    if not (r_means and g_means and b_means):
        return (0.0, 0.0, 0.0), (1.0, 1.0, 1.0)

    mean = (float(np.mean(r_means)), float(np.mean(g_means)), float(np.mean(b_means)))
    std = (
        float(np.std(r_means) or 1.0),
        float(np.std(g_means) or 1.0),
        float(np.std(b_means) or 1.0),
    )
    return mean, std


def jitter_and_normalize(img, mean, std):
    """Add small uniform noise then normalize per channel (BGR input/output)."""
    img_f = img.astype(np.float32)
    noise = np.random.uniform(-0.5, 0.5, img_f.shape).astype(np.float32)
    img_noisy = img_f + noise

    b, g, r = cv2.split(img_noisy)
    r_n = (r - mean[0]) / std[0]
    g_n = (g - mean[1]) / std[1]
    b_n = (b - mean[2]) / std[2]

    return cv2.merge([b_n, g_n, r_n])


def read_random_pixel(img):
    """Return ((x,y), pixel) for a random pixel in an HxWx3 array."""
    h, w, _ = img.shape
    x = random.randint(0, w - 1)
    y = random.randint(0, h - 1)
    return (x, y), img[y, x]


# =======================
# MAIN PIPELINE
# =======================
def process_folder(input_dir, output_dir):
    """Batch normalize images and save as .npz (RGB)."""
    os.makedirs(output_dir, exist_ok=True)
    batches = get_batches(input_dir)

    for batch_name, paths in batches.items():
        mean, std = calculate_batch_stats(paths)
        print(f"{batch_name}: {len(paths)} images")

        for p in paths:
            img = cv2.imread(p)
            if img is None:
                continue

            out = jitter_and_normalize(img, mean, std)

            # Optional sanity check: print one random pixel
            coord, pixel = read_random_pixel(out)
            base = os.path.splitext(os.path.basename(p))[0]
            print(f"  {base}.npz | pixel@{coord} = {pixel}")

            # Save compressed .npz with RGB channel order
            npz_path = os.path.join(str(output_dir), base + ".npz")
            out_rgb = out[:, :, ::-1]  # BGR -> RGB
            np.savez_compressed(npz_path, image=out_rgb)


if __name__ == "__main__":
    process_folder(INPUT_DIR, OUTPUT_DIR)